Small

In [ ]:
# ============================================================
# HAZMAT VRP FINAL (ROBUST VERSION)
# ============================================================

import pandas as pd
import pulp as pl
import math

# ============================================================
# 1. OD MATRIX LADEN
# ============================================================

od_df = pd.read_csv(
    r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Data\OperationsResearch\od_matrix_hazmat_small.csv"
)

od_df = od_df[od_df["profile"] == "safest"]

print("Spalten:", od_df.columns)

# ============================================================
# 2. DATENSTRUKTUREN (ROBUST!)
# ============================================================

dist, risk, time = {}, {}, {}
nodes = set()

for _, row in od_df.iterrows():

    i, j = row["from"], row["to"]

    d = row.get("dist_km", None)
    r = row.get("risk_cost", 0)
    t = row.get("time_min", None)

    # NaN / inf entfernen
    if pd.isna(d) or pd.isna(t) or math.isinf(d) or math.isinf(t):
        continue

    if pd.isna(r) or math.isinf(r):
        r = 0

    dist[(i, j)] = float(d)
    risk[(i, j)] = float(r)
    time[(i, j)] = float(t)

    nodes.add(i)
    nodes.add(j)

nodes = list(nodes)

print(f"Nodes: {len(nodes)} | Arcs: {len(dist)}")

# ============================================================
# FEHLENDE KANTEN (Penalty)
# ============================================================

max_dist = max(dist.values())
max_risk = max(risk.values()) if len(risk) > 0 else 1
max_time = max(time.values())

BIG_PENALTY = 2

for i in nodes:
    for j in nodes:
        if i != j and (i, j) not in dist:

            dist[(i, j)] = max_dist * BIG_PENALTY
            risk[(i, j)] = max_risk * BIG_PENALTY
            time[(i, j)] = max_time * BIG_PENALTY

# safety check
print("NaN Check Dist:", any(pd.isna(v) for v in dist.values()))
print("NaN Check Time:", any(pd.isna(v) for v in time.values()))

# ============================================================
# 3. SETS
# ============================================================

DEPOT = "DEPOT"
customers = [n for n in nodes if n != DEPOT]

vehicles = ["Truck1", "Truck2", "Truck3"]

# ============================================================
# 4. PARAMETER
# ============================================================

Demand = {c: 2000 for c in customers}

Cap = {
    "Truck1": 10000,
    "Truck2": 12000,
    "Truck3": 8000
}

FixCost = {v: 300 for v in vehicles}

ServiceTime = {c: 5 for c in customers}

MAX_TIME = 800

w_cost = 0.3
w_risk = 0.7

# ============================================================
# 5. MODELL
# ============================================================

model = pl.LpProblem("Hazmat_VRP_Robust", pl.LpMinimize)

x = pl.LpVariable.dicts(
    "x",
    [(i, j, v) for i in nodes for j in nodes if i != j for v in vehicles],
    cat="Binary"
)

T = pl.LpVariable.dicts(
    "T",
    [(i, v) for i in customers for v in vehicles],
    lowBound=0
)

U = pl.LpVariable.dicts(
    "U",
    customers,
    lowBound=0,
    upBound=len(customers)
)

# ============================================================
# 6. ZIELFUNKTION
# ============================================================

travel_cost = pl.lpSum(
    (w_cost * dist[(i, j)] + w_risk * risk[(i, j)]) * x[(i, j, v)]
    for i in nodes for j in nodes if i != j for v in vehicles
)

fixed_cost = pl.lpSum(
    FixCost[v] * pl.lpSum(x[(DEPOT, j, v)] for j in customers)
    for v in vehicles
)

model += travel_cost + fixed_cost

# ============================================================
# 7. CONSTRAINTS
# ============================================================

# Jeder Kunde genau einmal
for j in customers:
    model += pl.lpSum(
        x[(i, j, v)]
        for i in nodes if i != j for v in vehicles
    ) == 1

# Fluss
for v in vehicles:
    for h in customers:
        model += (
            pl.lpSum(x[(i, h, v)] for i in nodes if i != h) ==
            pl.lpSum(x[(h, j, v)] for j in nodes if j != h)
        )

# Depot
for v in vehicles:
    model += (
        pl.lpSum(x[(DEPOT, j, v)] for j in customers) ==
        pl.lpSum(x[(i, DEPOT, v)] for i in customers)
    )

    model += pl.lpSum(x[(DEPOT, j, v)] for j in customers) <= 1

# Kapazität
for v in vehicles:
    model += pl.lpSum(
        Demand[j] *
        pl.lpSum(x[(i, j, v)] for i in nodes if i != j)
        for j in customers
    ) <= Cap[v]

# ============================================================
# 8. ZEIT
# ============================================================

BIG_M = 100000

# Start
for v in vehicles:
    for j in customers:
        model += (
            T[(j, v)] >= time[(DEPOT, j)]
            - BIG_M * (1 - x[(DEPOT, j, v)])
        )

# Kunde → Kunde
for v in vehicles:
    for i in customers:
        for j in customers:
            if i != j:
                model += (
                    T[(j, v)] >=
                    T[(i, v)] + ServiceTime[i] + time[(i, j)]
                    - BIG_M * (1 - x[(i, j, v)])
                )

# Rückkehr
for v in vehicles:
    for i in customers:
        model += (
            T[(i, v)] + ServiceTime[i] + time[(i, DEPOT)]
            <= MAX_TIME + BIG_M * (1 - x[(i, DEPOT, v)])
        )

# Max Tourzeit
for v in vehicles:
    model += pl.lpSum(
        time[(i, j)] * x[(i, j, v)]
        for i in nodes for j in nodes if i != j
    ) <= MAX_TIME

# ============================================================
# 9. MTZ
# ============================================================

for i in customers:
    for j in customers:
        if i != j:
            model += (
                U[i] - U[j]
                + len(customers) * pl.lpSum(x[(i, j, v)] for v in vehicles)
                <= len(customers) - 1
            )

# ============================================================
# 10. SOLVER
# ============================================================

print("Starte Optimierung...")

solver = pl.PULP_CBC_CMD(msg=1, timeLimit=300, gapRel=0.02)

model.solve(solver)

print("\nStatus:", pl.LpStatus[model.status])
print("Objective:", pl.value(model.objective))

# ============================================================
# 11. ROUTEN
# ============================================================

def build_route(v):
    route = [DEPOT]
    current = DEPOT
    visited = set()

    while True:
        next_nodes = [
            j for j in customers + [DEPOT]
            if j != current and pl.value(x[(current, j, v)]) is not None and pl.value(x[(current, j, v)]) > 0.5
        ]

        if not next_nodes:
            break

        nxt = next_nodes[0]
        route.append(nxt)

        if nxt == DEPOT:
            break

        if nxt in visited:
            route.append("SUBTOUR")
            break

        visited.add(nxt)
        current = nxt

    return route

print("\nROUTEN:")

for v in vehicles:
    r = build_route(v)

    if len(r) > 1:
        print(f"{v}: {' -> '.join(r)}")
    else:
        print(f"{v}: nicht genutzt")

c:\Users\j.beckmann\AppData\Local\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\j.beckmann\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Spalten: Index(['from', 'to', 'from_type', 'to_type', 'profile', 'dist_km', 'time_min',
       'risk_cost', 'road_penalty_cost', 'objective', 'path_nodes',
       'reachable', 'path_signature', 'used_nodes', 'used_coords',
       'highway_km', 'used_nodes_str', 'used_coords_str', 'highway_km_str'],
      dtype='str')
Nodes: 11 | Arcs: 91
NaN Check Dist: False
NaN Check Time: False
Starte Optimierung...


In [ ]:
# ============================================================
# PLAUSIBILITÄTSCHECKS
# ============================================================

print("\n--- PLAUSIBILITÄTSCHECKS ---")

# 1. Distanz > 0
dist_values = list(dist.values())
print("Min Distanz:", min(dist_values))
print("Max Distanz:", max(dist_values))

if any(d <= 0 for d in dist_values):
    print("Ungültige Distanzen (<=0 gefunden)")
else:
    print("Alle Distanzen positiv")

# ------------------------------------------------------------

# 2. Zeit konsistent mit Distanz (Geschwindigkeit)
speeds = []

for (i, j) in dist:
    d = dist[(i, j)]       # km
    t = time[(i, j)]       # Minuten
    
    if t > 0:
        speed = d / (t / 60)  # km/h
        speeds.append(speed)

print(f"Geschwindigkeit: min={min(speeds):.1f}, max={max(speeds):.1f} km/h")

if min(speeds) < 5 or max(speeds) > 130:
    print("Unplausible Geschwindigkeit")
else:
    print("Geschwindigkeit plausibel")

# ------------------------------------------------------------

# 3. Risiko-Werte prüfen
risk_values = list(risk.values())

print("Min Risiko:", min(risk_values))
print("Max Risiko:", max(risk_values))

if any(r < 0 for r in risk_values):
    print("Negatives Risiko gefunden")
else:
    print("Risiko plausibel")

# ------------------------------------------------------------

# 4. Nachfrage vs Kapazität
total_demand = sum(Demand.values())
total_capacity = sum(Cap[v] for v in vehicles)

print("Gesamtnachfrage:", total_demand)
print("Gesamtkapazität:", total_capacity)

if total_demand > total_capacity:
    print("Nachfrage > Kapazität → Problem infeasible")
else:
    print("Kapazität ausreichend")

# ------------------------------------------------------------

# 5. Zeitfenster grob machbar?
max_trip_time = max(time.values())

print("Max Einzelkante Zeit:", max_trip_time)
print("MAX_TIME:", MAX_TIME)

if max_trip_time > MAX_TIME:
    print("Einzelverbindung länger als MAX_TIME")
else:
    print("Zeitstruktur plausibel")

# ------------------------------------------------------------

# 6. Erreichbarkeit prüfen (wichtiger Check!)
unreachable = od_df[od_df["reachable"] == False]

if len(unreachable) > 0:
    print(f"{len(unreachable)} nicht erreichbare Relationen")
else:
    print("Alle Relationen erreichbar")

# ------------------------------------------------------------

# 7. Depot erreichbar?
for c in customers:
    if (DEPOT, c) not in dist or (c, DEPOT) not in dist:
        print(f"Depot nicht verbunden mit {c}")

print("Depot-Verbindungen geprüft")


--- PLAUSIBILITÄTSCHECKS ---
Min Distanz: 27.056752126615446
Max Distanz: 542.4528392498312
✅ Alle Distanzen positiv
Geschwindigkeit: min=50.7, max=78.7 km/h
✅ Geschwindigkeit plausibel
Min Risiko: 0.0151286401754987
Max Risiko: 0.2336206367246552
✅ Risiko plausibel
Gesamtnachfrage: 20000
Gesamtkapazität: 30000
✅ Kapazität ausreichend
Max Einzelkante Zeit: 435.8262086732074
MAX_TIME: 800
✅ Zeitstruktur plausibel
⚠️ 19 nicht erreichbare Relationen
✅ Depot-Verbindungen geprüft


Hazmat VRP mit Reload, Risiko und Zeitrestriktionen

In [ ]:
import pandas as pd
import pulp as pl
import math

# ============================================================
# 1. OD MATRIX LADEN
# ============================================================

od_df = pd.read_csv(
    r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Data\OperationsResearch\od_matrix_hazmat_small.csv"
)

od_df = od_df[od_df["profile"] == "safest"]

# ============================================================
# 2. DATENSTRUKTUREN
# ============================================================

dist, risk, time = {}, {}, {}
nodes = set()

for _, row in od_df.iterrows():

    i, j = row["from"], row["to"]

    d = row.get("dist_km", None)
    r = row.get("risk_cost", 0)
    t = row.get("time_min", None)

    if pd.isna(d) or pd.isna(t) or math.isinf(d) or math.isinf(t):
        continue

    if pd.isna(r) or math.isinf(r):
        r = 0

    dist[(i, j)] = float(d)
    risk[(i, j)] = float(r)
    time[(i, j)] = float(t)

    nodes.add(i)
    nodes.add(j)

nodes = list(nodes)

# ============================================================
# FEHLENDE KANTEN
# ============================================================

max_dist = max(dist.values())
max_risk = max(risk.values()) if len(risk) > 0 else 1
max_time = max(time.values())

BIG_PENALTY = 2

for i in nodes:
    for j in nodes:
        if i != j and (i, j) not in dist:
            dist[(i, j)] = max_dist * BIG_PENALTY
            risk[(i, j)] = max_risk * BIG_PENALTY
            time[(i, j)] = max_time * BIG_PENALTY

# ============================================================
# SETS
# ============================================================

DEPOT = "DEPOT"
customers = [n for n in nodes if n != DEPOT]

vehicles = ["Truck1", "Truck2", "Truck3"]

# ============================================================
# PARAMETER
# ============================================================

Demand = {c: 2000 for c in customers}

Cap = {
    "Truck1": 10000,
    "Truck2": 12000,
    "Truck3": 8000
}

FixCost = {v: 300 for v in vehicles}

ServiceTime = {c: 5 for c in customers}

MAX_TIME = 800

w_cost = 0.3
w_risk = 0.7

# ============================================================
# MODELL
# ============================================================

model = pl.LpProblem("Hazmat_VRP_Reload", pl.LpMinimize)

# Routing
x = pl.LpVariable.dicts(
    "x",
    [(i, j, v) for i in nodes for j in nodes if i != j for v in vehicles],
    cat="Binary"
)

# Zeit
T = pl.LpVariable.dicts(
    "T",
    [(i, v) for i in customers for v in vehicles],
    lowBound=0
)

# Subtour
U = pl.LpVariable.dicts(
    "U",
    customers,
    lowBound=0,
    upBound=len(customers)
)

# Load
Load = pl.LpVariable.dicts(
    "Load",
    [(i, v) for i in nodes for v in vehicles],
    lowBound=0
)

# ============================================================
# ZIELFUNKTION
# ============================================================

travel_cost = pl.lpSum(
    (w_cost * dist[(i, j)] + w_risk * risk[(i, j)]) * x[(i, j, v)]
    for i in nodes for j in nodes if i != j for v in vehicles
)

fixed_cost = pl.lpSum(
    FixCost[v] * pl.lpSum(x[(DEPOT, j, v)] for j in customers)
    for v in vehicles
)

model += travel_cost + fixed_cost

# ============================================================
# CONSTRAINTS
# ============================================================

# Jeder Kunde genau einmal
for j in customers:
    model += pl.lpSum(
        x[(i, j, v)]
        for i in nodes if i != j for v in vehicles
    ) == 1

# Fluss
for v in vehicles:
    for h in customers:
        model += (
            pl.lpSum(x[(i, h, v)] for i in nodes if i != h) ==
            pl.lpSum(x[(h, j, v)] for j in nodes if j != h)
        )

# Depot: mehrere Starts erlaubt ✅
for v in vehicles:
    model += (
        pl.lpSum(x[(DEPOT, j, v)] for j in customers) ==
        pl.lpSum(x[(i, DEPOT, v)] for i in customers)
    )

# ============================================================
# LOAD LOGIK (WICHTIG)
# ============================================================

BIG_M = 100000

for v in vehicles:
    # Start im Depot: volle Ladung
    model += Load[(DEPOT, v)] == Cap[v]

    for i in nodes:
        for j in customers:
            if i != j:
                # Verbrauch bei Kunde
                model += (
                    Load[(j, v)] <= Load[(i, v)] - Demand[j]
                    + BIG_M * (1 - x[(i, j, v)])
                )

                # Nicht negativ
                model += Load[(j, v)] >= 0

    # Reload im Depot
    for i in customers:
        model += (
            Load[(DEPOT, v)] >= Load[(i, v)]
            + BIG_M * (x[(i, DEPOT, v)] - 1)
        )

# ============================================================
# ZEIT
# ============================================================

for v in vehicles:
    for j in customers:
        model += (
            T[(j, v)] >= time[(DEPOT, j)]
            - BIG_M * (1 - x[(DEPOT, j, v)])
        )

for v in vehicles:
    for i in customers:
        for j in customers:
            if i != j:
                model += (
                    T[(j, v)] >=
                    T[(i, v)] + ServiceTime[i] + time[(i, j)]
                    - BIG_M * (1 - x[(i, j, v)])
                )

for v in vehicles:
    for i in customers:
        model += (
            T[(i, v)] + ServiceTime[i] + time[(i, DEPOT)]
            <= MAX_TIME + BIG_M * (1 - x[(i, DEPOT, v)])
        )

for v in vehicles:
    model += pl.lpSum(
        time[(i, j)] * x[(i, j, v)]
        for i in nodes for j in nodes if i != j
    ) <= MAX_TIME

# ============================================================
# MTZ
# ============================================================

for i in customers:
    for j in customers:
        if i != j:
            model += (
                U[i] - U[j]
                + len(customers) * pl.lpSum(x[(i, j, v)] for v in vehicles)
                <= len(customers) - 1
            )

# ============================================================
# SOLVER
# ============================================================

solver = pl.PULP_CBC_CMD(msg=1, timeLimit=300, gapRel=0.02)

model.solve(solver)

print("\nStatus:", pl.LpStatus[model.status])
print("Objective:", pl.value(model.objective))

# ============================================================
# ROUTEN
# ============================================================


def extract_routes_for_vehicle(v):
    routes = []

    # aktive Kanten sammeln
    edges = [
        (i, j)
        for i in nodes for j in nodes if i != j
        if pl.value(x[(i, j, v)]) is not None and pl.value(x[(i, j, v)]) > 0.5
    ]

    # Nachfolger-Mapping
    successors = {}
    for (i, j) in edges:
        if i not in successors:
            successors[i] = []
        successors[i].append(j)

    # alle Touren vom Depot starten
    while True:

        if DEPOT not in successors or len(successors[DEPOT]) == 0:
            break

        route = [DEPOT]
        current = DEPOT

        while True:
            if current not in successors or len(successors[current]) == 0:
                break

            nxt = successors[current].pop(0)
            route.append(nxt)

            if nxt == DEPOT:
                break

            current = nxt

        if len(route) > 2:
            routes.append(route)

    return routes


print("\nROUTEN:")

for v in vehicles:
    routes = extract_routes_for_vehicle(v)

    if len(routes) == 0:
        print(f"{v}: nicht genutzt")
    else:
        for idx, r in enumerate(routes, 1):
            print(f"{v} Tour {idx}: {' -> '.join(r)}")


Status: Optimal
Objective: 1518.7118794120433

ROUTEN:
Truck1 Tour 1: DEPOT -> C2 -> C9 -> DEPOT
Truck2 Tour 1: DEPOT -> C10 -> C5 -> C4 -> C3 -> C1 -> DEPOT
Truck3 Tour 1: DEPOT -> C7 -> C6 -> C8 -> DEPOT
